In [ ]:
# Open the new type of netcdf file with parameters and spectra
# A. Lefauve, 2026

import h5py
path = "<PROJECT_ROOT>/R1P7/001_Final/R1P7.nc"
f = h5py.File(path, "r")

print("Top-level keys:", list(f.keys()))
print("Top-level keys:", list(f.keys()))

def walk(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(name, obj.shape, obj.dtype)

f.visititems(walk)

In [ ]:
# Export to Excel
import h5py
import pandas as pd
import os
!pip install openpyxl
path = "<PROJECT_ROOT>/R1P1/001_Final/R1P1.nc"

# extract just the filename (R1P1.nc → R1P1 → R1P1.xlsx)
filename = os.path.basename(path)            # R1P1.nc
name = os.path.splitext(filename)[0]         # R1P1
output_path = os.path.join(os.path.dirname(path), name + ".xlsx")

print(output_path)  # sanity check

with h5py.File(path, "r") as f:
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for key in f.keys():
            data = f[key][()].flatten()
            df = pd.DataFrame({key: data})
            df.to_excel(writer, sheet_name=key[:31], index=False)

print(f"Saved to {output_path}")

In [ ]:
# Now group all .nc in one directory and export xlsx

import h5py
import pandas as pd
import os
import shutil

folders = ["R1P1", "R1P7", "R1P50", "R4P1", "R4P7", "R4P50", "R6P1", "R6P7", "R6P50","R8P1", "R8P7","R10P1", "R10P7" ]  # your list
base_path = "<PROJECT_ROOT>"

# destination folder
dest_path = "<PROJECT_ROOT>/ALL_OUTPUTS"
os.makedirs(dest_path, exist_ok=True)

for folder in folders:
    source_folder = os.path.join(base_path, folder, "001_Final")

    if not os.path.exists(source_folder):
        print(f"Skipping {folder}: path not found")
        continue

    for file in os.listdir(source_folder):
        if file.endswith(".nc"):

            source_file = os.path.join(source_folder, file)
            dest_file = os.path.join(dest_path, file)

            print(f"\nProcessing {source_file}")

            # copy the .nc file
            shutil.copy2(source_file, dest_file)

            # output Excel file in destination folder
            name = os.path.splitext(file)[0]
            output_path = os.path.join(dest_path, name + ".xlsx")

            try:
                with h5py.File(source_file, "r") as f:
                    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

                        # --- first sheet: all scalar parameters ---
                        param_names = []
                        param_values = []

                        for key in f.keys():
                            data = f[key][()]
                            if data.size == 1:
                                param_names.append(key)
                                param_values.append(data.item())

                        params_df = pd.DataFrame({
                            "parameter": param_names,
                            "value": param_values
                        })
                        params_df.to_excel(writer, sheet_name="parameters", index=False)

                        # --- other sheets: non-scalar datasets only ---
                        for key in f.keys():
                            data = f[key][()]
                            if data.size == 1:
                                continue

                            data = data.flatten()
                            sheet_name = key[:31].replace("/", "_")
                            df = pd.DataFrame({key: data})
                            df.to_excel(writer, sheet_name=sheet_name, index=False)

                print(f"Saved → {output_path}")

            except Exception as e:
                print(f"Failed on {source_file}: {e}")

In [ ]:
# NO LONGER NEEDED: New code to open the new type of netcdf file sent by Steve mid January 2025
import h5py

path = "<PROJECT_ROOT>/R1P1/SpectraAdrien/Stats/R1P1.nc"
f = h5py.File(path, "r")

print("Top-level keys:", list(f.keys()))

def walk(name, obj):
    if isinstance(obj, h5py.Dataset):
        print(name, obj.shape, obj.dtype)

f.visititems(walk)

In [ ]:
# NO LONGER NEEDED: New code to inspect the new type of netcdf file created by Steve late January 2025 
from scipy.io import netcdf_file

path = "<PROJECT_ROOT>/R1P1/001_Final/Spectra/mespec_1100.000000.nc"
nc = netcdf_file(path, "r")

print("Dimensions:")
for k, n in nc.dimensions.items():
    print(f"  {k:20s} {n}")

print("\nVariables:")
for k, v in nc.variables.items():
    # v.typecode() is the scipy netcdf dtype indicator (like 'f', 'd', 'i')
    print(f"  {k:20s} shape={v.shape} typecode={v.typecode()} dims={v.dimensions}")

print("\nGlobal attributes:")
for a in nc._attributes:
    print(f"  {a} = {getattr(nc, a)}")